# Continuous ODE Adjoint Optimization for OMWU
This notebook demonstrates how to use the `ODEAdjointOptimizer` to find payoff matrices that cause the Optimistic Multiplicative Weights Update (OMWU) algorithm to suffer maximum asymptotic regret in a 2-player game, using PyTorch's `torchdiffeq` for continuous-time backpropagation.\n

In [1]:
import sys
sys.path.append("..")

import numpy as np
import torch
import matplotlib.pyplot as plt

from src.config.schemas import ExperimentConfig, GameConfig, DynamicConfig, ExecutionConfig, CMAESConfig
from src.engine.ode_optimizer import ODEAdjointOptimizer
from src.engine.runner import ExperimentRunner
from src.engine.statistics import load_experiment_stats

# Set formatting for plots
plt.style.use('ggplot')

## 1. Configure the ODE Optimizer
We configure a 2x2 game base template and set the objective to `envelope_trend_log` to hunt for cyclic regret envelopes.
Note that for the ODE optimizer, `execution.total_steps` acts as the integration time `T`, and `steps_per_call` acts as `N_steps` (the number of timepoints to evaluate).\n

[tensor([[-0.4312, -0.3113],
         [-0.0567, -0.3249]], dtype=torch.float64),
 tensor([[-0.0921, -0.1353],
         [ 0.1871,  0.1918]], dtype=torch.float64)]

In [2]:
# Setup configuration
config = ExperimentConfig(
    name="ode_omwu_demo",
    game=GameConfig(
        generator="custom",
        utility_range=(-1.0, 1.0),
        payoffs=[
            [[1.0, 0.5], [0.5, 1.0]], # Player 1 base
            [[1.0, -1.0], [0.5, 1.0]], # Player 2 base
        ]
    ),
    dynamic=DynamicConfig(
        algorithm="omwu", 
        # eta=0.05,
        logit_penalty_threshold=25,
        logit_penalty_norm=2,
        logit_penalty_mode="centered"
    ),
    execution=ExecutionConfig(
        total_steps=50,      # T: Total integration time for ODE
        steps_per_call=500,  # N_steps: Number of evaluation points
        device="cuda" if torch.cuda.is_available() else "cpu",
        dtype="float64",     # ODE adjoint precision
        compile=False        # ODEint handles its own compilation internally
    ),
    cmaes=CMAESConfig(       # We borrow CMAESConfig for the objective parameters
        objective_type="envelope_trend_log",
        T1_ratio=0.8,
    )
)

# Initialize ODE Adjoint Optimizer
optimizer = ODEAdjointOptimizer(
    action_sizes=[2, 2], eta=config.dynamic.eta, N_steps=config.execution.steps_per_call,)

## 2. Run the Optimization
We run the ODE optimizer for a set number of epochs. The progress bar will natively show the Loss and Penalty decreasing.\n

In [4]:
# Run optimization
best_payoffs, loss_history, penalty_history, final_states, final_t = optimizer.optimize(epochs=200)

print("Optimization Complete!")
print("\nWorst-case Payoff Matrix Player 1:")
print(np.round(best_payoffs[0].detach().cpu().numpy(), 4))
print("\nWorst-case Payoff Matrix Player 2:")
print(np.round(best_payoffs[1].detach().cpu().numpy(), 4))

Output()

Optimization Complete!

Worst-case Payoff Matrix Player 1:
[[-0.4615 -0.6644]
 [ 0.459   0.4853]]

Worst-case Payoff Matrix Player 2:
[[-0.5056  0.5456]
 [-0.5656  0.6676]]


## 3. Plot Optimization Progress
Visualize the gradient descent loss and penalty curves across the epochs.\n

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(loss_history, label="Total Loss", color="purple")
axes[0].set_title("ODE Adjoint Loss over Epochs")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(penalty_history, label="Logit Penalty", color="orange")
axes[1].set_title("Logit Penalty over Epochs")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Penalty")
axes[1].legend()

plt.tight_layout()
plt.show()\n

## 4. Simulate the Worst-Case Game
We plug the discovered worst-case matrices back into a standard `ExperimentRunner` and run a discrete high-fidelity simulation to see the exact dynamics.\n

In [ ]:
import copy

# Create a validation config with the worst-case matrices
val_config = copy.deepcopy(config)
val_config.parent_session_id = session_id
val_config.dynamic.eta = 0.05
val_config.game.payoffs = [p.detach().cpu().numpy().tolist() for p in best_payoffs]
val_config.execution.batch_size = 1
val_config.execution.total_steps = 200000
val_config.execution.steps_per_call = 500
val_config.execution.dtype = "float64" # Discrete simulation in float64 for exact precision
val_config.name = "worst_case_omwu_ode"

# Run the single simulation
runner = ExperimentRunner(val_config)
summary = runner.run()

print(f"Validation Run Complete. Session ID: {summary['session_id']}")\n

## 5. Plot the Trajectories
Extract the cumulative regret and strategy probabilities from the discrete simulation to visualize the chaotic or non-converging dynamics discovered by the ODE optimizer.\n

In [ ]:
from src.engine.statistics import unpack_stats
from src.utils.visualization import plot_static_trajectories

# Load and unpack the recorded statistics from disk
stats_data = load_experiment_stats(output_dir='outputs', session_id=summary['session_id'])
steps, cum_regrets, strats, logits, instant_payoffs = unpack_stats(stats_data)

# Plot the static N-player trajectories
cum_action_payoffs = [torch.cumsum(p, dim=0) for p in instant_payoffs]
cum_expected_payoffs = [torch.cumsum(torch.sum(s * p, dim=-1), dim=0) for s, p in zip(strats, instant_payoffs)]
fig, axes = plot_static_trajectories(steps, cum_regrets, strats, title_prefix="ODE OMWU", plot_all_actions=False, start_step=0, end_step=None, cum_action_payoffs=cum_action_payoffs, cum_expected_payoffs=cum_expected_payoffs, logits=logits)
plt.show()\n